# Anime Data Analysis

This notebook explores an anime dataset, performing feature engineering to extract episode counts and airing durations, followed by an analysis of scores and rankings.

### 1. Imports and Data Loading

We start by importing the necessary libraries and loading the `anime.csv` dataset.

In [ ]:
import pandas as pd
from dateutil.relativedelta import relativedelta
from datetime import datetime

# Load the dataset
anime_df = pd.read_csv('anime.csv')
anime_df.head()

### 2. Feature Engineering: Extracting Episode Count

The episode count is currently embedded within the `Title` string in parentheses. We define a function to extract this information and convert it to an integer type.

In [ ]:
def eps_extract(title):
    """Extracts the number of episodes from the title string."""
    check = False
    data = ''
    for i in title:
        if i == ')':
            check = False
            return data
        if check:
            data = data + i
        if i == '(':
            check = True
    return data

# Apply extraction and cleaning
anime_df['Episodes'] = anime_df["Title"].apply(eps_extract)
anime_df["Episodes" ] = anime_df["Episodes"].str.replace(' eps', '')

# Convert to integer
anime_df['Episodes'] = anime_df["Episodes"].astype(int)

print(f"Datatype of Episodes: {anime_df['Episodes'].dtype}")
anime_df.head()

### 3. Feature Engineering: Extracting Airing Time Span

We also extract the string representing the airing period of each anime.

In [ ]:
def extract_time(title):
    """Extracts the time span suffix from the title."""
    data = ''
    for i in range(len(title)):
        if title[i] == ')':
            # Extract characters after the first closing parenthesis
            data = title[i+1:].strip()
            break
    return data

anime_df["Time"] = anime_df["Title"].apply(extract_time)
anime_df.head()

### 4. Data Analysis: Scoring and Popularity

In this section, we identify the top-performing anime based on their scores and find the ones with the most episodes.

In [ ]:
# Finding the highest score
max_score = anime_df["Score"].max()
highest_score_anime = anime_df[anime_df["Score"] == max_score]["Title"].iloc[0]
print(f"The anime with the highest score ({max_score}) is: {highest_score_anime}")

print("\nTop 5 Highest Scoring Anime:")
display(anime_df.nlargest(5, "Score"))

In [ ]:
# Highest episode count
highest_eps = anime_df["Episodes"].max()
highest_eps_anime = anime_df[anime_df["Episodes"] == highest_eps]
print(f"Anime with the highest episode count ({highest_eps}):")
display(highest_eps_anime)

print("\nTop 5 Anime by Episode Count:")
display(anime_df.nlargest(5, "Episodes"))

### 5. Advanced Analysis: Longest Running Anime

We calculate the total duration in months for each anime based on the extracted time span.

In [ ]:
def calculate_duration_months(time_str):
    """Calculates total months between two dates in 'Jan 2000' format."""
    try:
        start, end = time_str.split(' - ')
        start_date = datetime.strptime(start.strip(), '%b %Y')
        end_date = datetime.strptime(end.strip(), '%b %Y')
        r = relativedelta(end_date, start_date)
        # Including the start month in total count
        return r.years * 12 + r.months + 1
    except:
        return None

anime_df["Months"] = anime_df["Time"].apply(calculate_duration_months)
anime_df.sort_values("Months", ascending=False).head()